# Topic 4 - Statelessness

The API has no memory between calls. The conversation is a list **you** resend every turn,
and that resend is what makes cost grow on every turn even when the user asks nothing new.

This notebook proves that against the running local model, then measures the cost of doing it.

By the end I should be able to:

- show that a bare follow-up question is ambiguous to a stateless model
- show the same question answered once the history is resent
- measure input tokens as the history grows turn by turn
- implement a sliding window and a summarising compaction strategy
- explain what "cached prefix" and provider-side "threads" really do (and do not do)

## 0. Setup

One helper that calls the local OpenAI-compatible server. It turns thinking off so Qwen3
answers directly (otherwise the visible content can be empty), uses a generous timeout for
a slow local model, and fails loudly if the server returns nothing.

In [1]:
import hashlib
import os

import requests

try:
    import tiktoken

    _ENC = tiktoken.get_encoding("cl100k_base")

    def count_tokens(text: str) -> int:
        return len(_ENC.encode(text))

    BACKEND = "tiktoken / cl100k_base"
except Exception:  # noqa: BLE001 - notebook convenience
    def count_tokens(text: str) -> int:
        return max(1, round(len(text) / 4))

    BACKEND = "heuristic ~4 chars/token"

BASE_URL = os.getenv("LLM_BASE_URL", "http://127.0.0.1:8080/v1")
MODEL = os.getenv("LLM_MODEL", "Qwen/Qwen3-4B-GGUF:Q4_K_M")

SYSTEM = "You are a concise assistant. Answer in two sentences or fewer."


def chat(messages, temperature: float = 0.2, max_tokens: int = 200) -> str:
    payload = {
        "model": MODEL,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    resp = requests.post(f"{BASE_URL}/chat/completions", json=payload, timeout=300)
    resp.raise_for_status()
    message = resp.json()["choices"][0]["message"]
    content = message.get("content")
    if not isinstance(content, str) or not content.strip():
        raise ValueError(
            "The model returned empty content. For Qwen3, confirm the server supports "
            "chat_template_kwargs.enable_thinking=false."
        )
    return content.strip()

## 1. A model call is a pure function of its input

```
response = model(current_request)
```

If `current_request` does not contain the earlier conversation, the model cannot see it.
Nothing is stored server-side between two calls.

Ask a first question, then ask a follow-up that only makes sense with the first answer in
view. The follow-up is sent **alone** first, then again with the history resent.

In [2]:
first_question = "What is Java?"
follow_up = "What are its main advantages?"

# Turn 1: a normal question.
first_answer = chat(
    [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": first_question},
    ]
)
print("Turn 1")
print("  Q:", first_question)
print("  A:", first_answer)
print("-" * 72)

# Turn 2, stateless: send ONLY the follow-up (same system prompt, no history).
alone = chat(
    [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": follow_up},
    ]
)
print("Turn 2 - follow-up sent ALONE (stateless)")
print("  Q:", follow_up)
print("  A:", alone)
print("-" * 72)

# Turn 2, with history: resend turn 1 plus the follow-up.
with_history = chat(
    [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": first_question},
        {"role": "assistant", "content": first_answer},
        {"role": "user", "content": follow_up},
    ]
)
print("Turn 2 - same follow-up WITH the history resent")
print("  A:", with_history)
print()
print("The model did not 'remember' turn 1 in the second call. The third call simply")
print("included turn 1 in its input. The application owns the state.")

Turn 1
  Q: What is Java?
  A: Java is a high-level programming language known for its portability and platform independence. It is widely used for building applications across various platforms.
------------------------------------------------------------------------
Turn 2 - follow-up sent ALONE (stateless)
  Q: What are its main advantages?
  A: The product offers enhanced performance and reliability. It also provides cost-effective solutions for long-term use.
------------------------------------------------------------------------
Turn 2 - same follow-up WITH the history resent
  A: Java's main advantages include platform independence, robustness, and a vast ecosystem of libraries and tools. It is also known for its readability and strong type system.

The model did not 'remember' turn 1 in the second call. The third call simply
included turn 1 in its input. The application owns the state.


## 2. The application owns the state

The model never stores the conversation. The application appends each exchange and resends
the whole list. Build that loop and measure the input as it grows.

In [3]:
def render(history):
    return "".join(f"{role}: {content}\n" for role, content in history)


def build_history(turns: int):
    history = [("system", SYSTEM), ("user", first_question), ("assistant", first_answer)]
    for i in range(1, turns + 1):
        history.append(("user", f"Question {i}: tell me one thing about topic {i}."))
        history.append(
            ("assistant", f"Answer {i}: here is a short, concrete sentence about topic {i}.")
        )
    return history


history = build_history(10)

print(f"counting backend: {BACKEND}\n")
print(f"{'turn':>5}{'input tokens':>14}{'growth':>10}")
previous = 0
for turn in range(1, 11):
    partial = history[: 3 + 2 * turn]  # system + first exchange + 2 messages per turn
    n = count_tokens(render(partial))
    print(f"{turn:>5}{n:>14}{n - previous:>10}")
    previous = n

print()
print("Every turn re-sends everything before it, so the input keeps growing on a turn")
print("where the user asked nothing new.")

counting backend: tiktoken / cl100k_base

 turn  input tokens    growth
    1            83        83
    2           116        33
    3           149        33
    4           182        33
    5           215        33
    6           248        33
    7           281        33
    8           314        33
    9           347        33
   10           380        33

Every turn re-sends everything before it, so the input keeps growing on a turn
where the user asked nothing new.


In [4]:
full_history_tokens = count_tokens(render(history))
total_paid = sum(
    count_tokens(render(history[: 3 + 2 * turn])) for turn in range(1, 11)
)

print(f"final transcript size : {full_history_tokens:>6} tokens")
print(f"total input billed    : {total_paid:>6} tokens across 10 turns")
print(f"re-send overhead      : {total_paid / full_history_tokens:.1f}x the final transcript")

final transcript size :    380 tokens
total input billed    :   2315 tokens across 10 turns
re-send overhead      : 6.1x the final transcript


## 3. Lever 1 - a sliding window

Keep the system prompt and the last N messages. Cheap, predictable, and lossy: anything
that falls out of the window is gone.

In [5]:
def render_window(history, window_messages: int) -> str:
    system = [m for m in history if m[0] == "system"]
    rest = [m for m in history if m[0] != "system"]
    return render(system + rest[-window_messages:])


print(f"{'strategy':<28}{'input tokens':>14}")
print(f"{'full history':<28}{count_tokens(render(history)):>14}")
for n in (8, 4, 2):
    print(f"{'last ' + str(n) + ' messages':<28}{count_tokens(render_window(history, n)):>14}")
print()
print("The window bounds cost. The price is recall: ask about the first exchange and it is")
print("no longer in the request.")

strategy                      input tokens
full history                           380
last 8 messages                        147
last 4 messages                         81
last 2 messages                         48

The window bounds cost. The price is recall: ask about the first exchange and it is
no longer in the request.


## 4. Lever 2 - compaction by summarisation

Instead of dropping old turns, replace them with a short summary. Lossy in a different way:
details go, but the thread of the conversation survives. This one uses the real model.

In [6]:
older = history[3:-4]   # everything except the system prompt, first exchange, last 2 turns
recent = history[-4:]

summary = chat(
    [
        {
            "role": "system",
            "content": (
                "Summarise the conversation below in one short sentence. Keep only what a "
                "reader needs to follow the next turn."
            ),
        },
        {"role": "user", "content": render(older)},
    ],
    temperature=0.2,
    max_tokens=96,
)

compacted = [("system", SYSTEM), ("system", "Summary so far: " + summary)] + recent

print(f"{'strategy':<28}{'input tokens':>14}")
print(f"{'full history':<28}{count_tokens(render(history)):>14}")
print(f"{'window (last 4)':<28}{count_tokens(render_window(history, 4)):>14}")
print(f"{'summary + last 4':<28}{count_tokens(render(compacted)):>14}")
print()
print("Summary:", summary)
print()
print("Compaction trades detail for a bounded, informative request.")

strategy                      input tokens
full history                           380
window (last 4)                         81
summary + last 4                       112

Summary: The assistant has provided a concise answer to each of the eight questions, offering a single, concrete sentence about each specified topic.

Compaction trades detail for a bounded, informative request.


## 5. Things that look like memory but are not

Two features make the *cost* of statelessness smaller without making the API stateful.

**Cached prefixes.** If the front of the request is byte-identical to last time, the
provider can reuse its internal computation and charge a fraction of input price for those
tokens. You still send them. The API still has no memory.

**Provider-side threads / `previous_response_id`.** The provider stores the earlier messages
and re-assembles the prefix for you, so you send *less over the wire*. The model still
receives the full prefix; it is storage convenience, not model memory.

In [7]:
def prefix_text(history, n_messages: int) -> str:
    """Render the system prompt plus the first `n_messages` messages."""
    return render(history[: n_messages + 1])


def fingerprint(text: str) -> str:
    return hashlib.sha256(text.encode()).hexdigest()[:12]


h5 = build_history(5)
h12 = build_history(12)

first_6_of_5 = prefix_text(h5, 5)
first_6_of_12 = prefix_text(h12, 5)

print("fingerprint over first 6 messages (history of 5) :", fingerprint(first_6_of_5))
print("fingerprint over first 6 messages (history of 12):", fingerprint(first_6_of_12))
print("prefix identical as the conversation grew:", first_6_of_5 == first_6_of_12)
print()

changed_system = [("system", "You are a terse assistant.")] + h12[1:]
print(
    "after changing the system prompt, prefix still identical:",
    prefix_text(changed_system, 5) == first_6_of_12,
)
print()
print("A byte-identical prefix can be cached: the provider discounts re-computation of it.")
print("It is NOT memory: the API still receives the whole prefix, and any change to an")
print("early byte invalidates the cache from that point on.")

fingerprint over first 6 messages (history of 5) : 7179308cdb62
fingerprint over first 6 messages (history of 12): 7179308cdb62
prefix identical as the conversation grew: True

after changing the system prompt, prefix still identical: False

A byte-identical prefix can be cached: the provider discounts re-computation of it.
It is NOT memory: the API still receives the whole prefix, and any change to an
early byte invalidates the cache from that point on.


## 6. What I learned

1. Why does a follow-up question fail when sent alone?
2. Where does conversation state actually live, and who resends it?
3. Why does cost grow on a turn where the user asks nothing new?
4. What does a sliding window buy, and what does it cost?
5. How is compaction different from truncation?
6. Why is a cached prefix not the same thing as the API remembering?

## Topic 4 - Done When

I can narrate where conversation state lives, why cost grows every turn, and which strategy
(window, compaction, caching) I would reach for and why.

## References

- Chip Huyen - Notes on LLM Engineering
- Anthropic - prompt caching
- Topic 3 notebook (`context_window.ipynb`) for the shared input/output budget